# BCSE432E Reinforcement Learning: Practical Exam Study Guide

Built from your **exact 12 lab assignments** (Dr. G. Bharadwaja Kumar). Every lab uses the **same four blocks in the same order**. Only three slots change: `POLICY`, `ENVIRONMENT` and `EQUATION`. Variable names are the same everywhere (`state, action, next_state, reward, alpha, gamma, epsilon, Q, V`). Run cells top to bottom in Google Colab.

| Lab | Exact assignment | Algorithm | Libraries |
|-----|------------------|-----------|-----------|
| 1 | Choose advertisements with epsilon-greedy variants and compute regret (min 500 trials) | Bandit | NumPy |
| 2 | 10x10 stochastic grid world, Value Iteration, compare gamma = 0 and 1 | Planning | NumPy |
| 3 | Taxi fleet (7 zones), Policy Iteration, compare with Value Iteration | Planning | NumPy |
| 4 | Monte Carlo prediction: 3-state MDP and 4x4 grid world | Prediction | NumPy |
| 5 | Delivery drone on a 16x16 grid with battery, wind, 2 packages, Q-Learning | Control | NumPy |
| 6 | Traffic-light agent with SARSA, Poisson arrivals, vs fixed cycle, 5000 vs 10000 episodes | Control | NumPy |
| 7 | Deep Q-learning on CartPole | DQN | PyTorch, Gymnasium |
| 8 | TD-Gammon with TD learning, different hyper-parameters, 1-ply vs 2-ply | TD(lambda) + net | NumPy |
| 9 | REINFORCE on CartPole, compared with a Deep Q-Learning baseline | Policy gradient | PyTorch, Gymnasium |
| 10 | Elevator control with Actor-Critic (discrete + continuous actions) | Actor-Critic | PyTorch |
| 11 | Delivery vehicle fuel efficiency, Simple vs Advanced Actor-Critic | Actor-Critic / A2C | PyTorch |
| 12 | PPO on any real-world task (simulated environment allowed) | PPO | PyTorch |

**Exam routine (5 steps)**
1. Write the four block comments first (`# 1. SETUP` ... `# 4. OUTPUT`).
2. Fill SETUP with the numbers given in the question (alpha, gamma, episodes, thresholds). Copy them exactly.
3. Build the MDP/environment from the question's rules (states, actions, rewards, transition probabilities).
4. Write the loop, then fill POLICY, ENVIRONMENT, EQUATION.
5. Print exactly what the question asks for. Many questions also ask you to **write observations**: see the "Observation to write" box in each lab.

> **Honest notes on what was tested**: Labs 1 to 6 and Lab 8 were run and the printed results are quoted below. The neural-network labs (7, 9, 10, 11, 12) were written carefully and their custom environments were tested, but I could not install PyTorch in my sandbox, so run those once tonight. Lab 5's results are described honestly in its section.

## The Template (memorize this once)

```python
# 1. SETUP                      -> alpha, gamma, epsilon, episodes
# 2. STATES AND ACTIONS         -> states, goal, moves (actions), Q or V table (or networks)
# 3. TRAINING LOOP
for episode in range(episodes):
    state = start
    while state != goal:              # (or: for step in range(...), or: while not done)
        # POLICY                      -> how to choose the action
        # ENVIRONMENT                 -> next_state, reward
        # EQUATION                    -> the learning update   <-- changes per algorithm
        state = next_state
# 4. OUTPUT                     -> print the table / path / reward
```

**What goes in each slot**

| Slot | Tabular (Labs 1-6) | Neural (Labs 7-12) |
|------|--------------------|--------------------|
| SETUP | `alpha, gamma, epsilon, episodes` | `gamma, lr, episodes` (+ `epsilon` for DQN) |
| STATES AND ACTIONS | `Q = np.zeros(...)` or `V = np.zeros(...)` | `nn.Sequential(...)` networks + `optimizer` |
| POLICY | random / epsilon-greedy / `argmax` | `Categorical(actor(x)).sample()` or epsilon-greedy on `model(x)` |
| ENVIRONMENT | `next_state, reward` from a few lines or a `step()` function | `env.step(action)` from Gymnasium |
| EQUATION | one update line | a `loss`, then `zero_grad() / backward() / step()` |

**The equations (the only thing that really changes)**
```python
# Bandit        Q[action] += (reward - Q[action]) / N[action]                      no next state
# TD(0)         V[state]  += alpha * (reward + gamma * V[next_state] - V[state])
# Monte Carlo   V[state]  += (G - V[state]) / N[state]                              G = return of the whole episode
# Policy Iter.  V[s] = r + gamma * V[s2]  (evaluate), then policy[s] = argmax_a (r + gamma * V[s2])  (improve)
# Q-Learning    Q[s, a]   += alpha * (r + gamma * max(Q[s2]) - Q[s, a])
# SARSA         Q[s, a]   += alpha * (r + gamma * Q[s2, a2] - Q[s, a])
# DQN           loss = (Q(s, a) - (r + gamma * max Q_target(s2))) ** 2
# TD(lambda)    trace = gamma*lambda*trace + grad V(s);   w += alpha * (V(s2) - V(s)) * trace
# REINFORCE     loss = -(log_prob * G)
# Actor-Critic  adv = r + gamma * V(s2) - V(s);   loss = -log_prob * adv + adv ** 2
# A2C           adv = G - V(s);   loss = -log_prob * adv + 0.5 * adv ** 2 - 0.01 * entropy
# PPO           ratio = exp(new_log_prob - old_log_prob);   loss = -min(ratio * adv, clip(ratio, 0.8, 1.2) * adv)
```

**Symbols**: `alpha` = learning rate, `gamma` = discount (how much the future matters), `epsilon` = exploration probability, `G` = discounted return = `r1 + gamma*r2 + gamma^2*r3 + ...`.

## Cheat Sheet: Snippets You Must Be Able to Write From Memory

```python
# Epsilon-greedy (tabular)
if random.random() < epsilon:
    action = random.randint(0, n_actions - 1)         # explore
else:
    action = np.argmax(Q[state])                       # exploit

# Discounted returns (used in 4, 9, 11, 12)
G = 0
returns = []
for reward in reversed(rewards):
    G = reward + gamma * G
    returns.insert(0, G)

# Policy network + sampling (used in 9, 10, 11, 12)
actor = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 2), nn.Softmax(dim=-1))
dist = torch.distributions.Categorical(actor(x))
action = dist.sample()                                 # a tensor
log_prob = dist.log_prob(action)                       # used in the loss
env.step(action.item())                                # .item() converts tensor -> int

# The 3-line PyTorch update (always the same)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Gymnasium CartPole (note: reset() returns 2 things, step() returns 5 things)
env = gym.make("CartPole-v1")
state, _ = env.reset()
next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated
```

**Why these details matter**
- `.detach()` on a target or advantage stops gradients flowing into the wrong network. Forgetting it is the most common bug.
- `terminated` means the pole really fell (no future reward, so the target has no `gamma * V(next)` term). `truncated` means the 500-step time limit hit (the future was not zero, so do not treat it as terminal).
- Dividing states by a constant (`state / 100`) keeps network inputs small and training stable.

## Lab 1. Epsilon-Greedy Variants for Advertisements, with Regret

**Assignment**: choose which advertisement to show using epsilon-greedy variants and compute the regret for at least 500 trials.

**Theory**
- Each ad is an **arm** with a hidden click probability. Showing an ad is an **action**; click = reward 1.
- **Update (given in the question)**: `Q(A)_new = Q(A)_old + (1/N(A)) * (R - Q(A)_old)`. The `1/N` makes `Q` the exact average of the rewards seen.
- **Epsilon variants** (the formulas are given in the question). `t` is the trial number:

| Variant | Formula | Behaviour |
|---------|---------|-----------|
| Standard | `eps = 0.1` (constant) | explores forever, keeps paying a regret "tax" |
| Linear decay | `eps_start - (eps_start - eps_end)/T * t` | steady drop |
| Exponential decay | `eps_end + (eps_start - eps_end) * exp(-lambda * t)` | explores fast early, then settles |
| Step decay | `eps_start * drop ** floor(t / step_size)`, clipped at `eps_end` | drops at fixed intervals |
| Inverse time | `eps_start / (1 + decay_rate * t)` | slow tail |

- **Regret**: `R(T) = sum over t of (mu* - mu_at)`. `mu*` is the true click rate of the best ad, `mu_at` the true rate of the ad chosen at trial t. It measures how much reward was lost by not always picking the best ad. Low regret that flattens out = good.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy where epsilon comes from a function `epsilon_fn(t)` |
| ENVIRONMENT | reward 1 with probability `true_ctr[action]` |
| EQUATION | `Q[a] += (reward - Q[a]) / N[a]`, plus add `best_mu - true_ctr[action]` to the regret |

**Observation to write**: decaying epsilon reduces regret because the agent stops wasting trials on bad ads once it has learned. In my run the exponential and step variants had the lowest regret, standard (constant 0.1) was in the middle, and the slow linear decay had the highest because it explores for too long. Results vary between runs, so the code averages 20 runs.

**Common mistakes**: regret uses the *true* rates, not `Q`; `N` must be incremented before dividing.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

# 1. SETUP
trials = 1000                                   # minimum 500 trials
runs = 20                                       # repeat and average (bandits are noisy)
eps_start, eps_end = 1.0, 0.01

# 2. STATES AND ACTIONS
true_ctr = [0.05, 0.12, 0.08, 0.20, 0.10]       # hidden click probability of each ad
n_actions = len(true_ctr)
best_mu = max(true_ctr)                         # mu*: expected reward of the best ad

# EPSILON VARIANTS: each returns epsilon at trial t
def standard(t):    return 0.1
def linear(t):      return max(eps_end, eps_start - (eps_start - eps_end) / trials * t)
def exponential(t): return eps_end + (eps_start - eps_end) * np.exp(-0.005 * t)
def step_decay(t):  return max(eps_end, eps_start * 0.5 ** (t // 100))
def inverse(t):     return eps_start / (1 + 0.01 * t)

variants = {"Standard": standard, "Linear": linear, "Exponential": exponential,
            "Step": step_decay, "Inverse time": inverse}

# 3. TRAINING LOOP
def run_bandit(epsilon_fn):
    Q = np.zeros(n_actions)
    N = np.zeros(n_actions)
    regret = []
    total = 0
    for t in range(trials):
        # POLICY: epsilon-greedy with epsilon_t
        if random.random() < epsilon_fn(t):
            action = random.randint(0, n_actions - 1)
        else:
            action = np.argmax(Q)

        # ENVIRONMENT: click or no click
        reward = 1 if random.random() < true_ctr[action] else 0

        # EQUATION: Q_new = Q_old + (1/N) * (R - Q_old)
        N[action] += 1
        Q[action] += (reward - Q[action]) / N[action]

        # REGRET: mu* - mu of the chosen action
        total += best_mu - true_ctr[action]
        regret.append(total)
    return np.array(regret), Q

results = {}
for name, fn in variants.items():
    results[name] = np.mean([run_bandit(fn)[0] for _ in range(runs)], axis=0)

# 4. OUTPUT
print("Final cumulative regret after", trials, "trials (average of", runs, "runs):")
for name, curve in results.items():
    print(f"  {name:13s}: {curve[-1]:.2f}")
_, Q = run_bandit(exponential)
print("Estimated CTR (exponential run):", np.round(Q, 3), "| best ad:", np.argmax(Q))

for name, curve in results.items():
    plt.plot(curve, label=name)
plt.xlabel("Trial"); plt.ylabel("Cumulative regret"); plt.legend(); plt.show()

## Lab 2. Grid World MDP and Value Iteration (10x10)

**Assignment**: 10x10 grid, start (0,0), goal (9,9). Intended move happens with probability 0.8, otherwise a random other direction (0.2). Reward +10 on reaching the goal, -1 for every other move, `gamma = 0.9`, terminal state (9,9). Implement **Value Iteration** (threshold `1e-4`), print the optimal value function and optimal policy, then compute values for `gamma = 0` and `gamma = 1` and write observations.

**Theory**
- **Value Iteration** needs the full model `P(s' | s, a)` and reward. It applies the Bellman **optimality** equation as an update, sweeping over all states until the largest change `delta < theta`:
  `V(s) = max over a of  sum over s' of  P(s'|s,a) * [ r + gamma * V(s') ]`
- Afterwards the policy is `argmax` of the same expression.
- Terminal state: no actions, `V = 0`, so the `+10` for reaching it propagates backward through the grid.

**How the MDP is represented in the code (this is the "Represent the MDP" part of the question)**
- State space: 100 cells numbered `row * 10 + col`. Action space: 0 Up, 1 Down, 2 Left, 3 Right.
- Transition probabilities: dictionary `P[state][action] = [(probability, next_state, reward), ...]`. The intended direction has probability 0.8 and each of the other three has `0.2/3`. A move off the grid leaves the agent in place.
- Reward function: `+10` if `next_state == goal`, else `-1`.
- *Assumption*: "random other direction" is modelled as one of the other three directions (off-grid means staying). Say this assumption in your answer.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | none while learning (it is derived from `V` at the end) |
| ENVIRONMENT | the model `P` (known) |
| EQUATION | `V[s] = max_a sum(p * (r + gamma * V[s2]))` |

**Observations to write (from my run)**
- `gamma = 0.9`: converges in about 32 sweeps. Values rise smoothly toward the goal (about -8 at the start, about +9 next to the goal). The policy sends the agent down and right.
- `gamma = 0` (completely myopic): converges in 2 sweeps. Every state is worth just its immediate reward, about `-1`, except the two cells next to the goal (about `7.8`). Far states have no idea where the goal is, so their policy is arbitrary (all actions tie).
- `gamma = 1` (no discounting): converges in about 37 sweeps. Values are the expected total undiscounted return, so far states look worse (about -12.8 at the start) and near-goal states a bit better (about +9.5). It still converges because the goal is reachable and terminal. With a continuing task and `gamma = 1` the sums could diverge.
- General: a larger `gamma` makes the agent more far-sighted and takes more sweeps to converge.

In [ ]:
import numpy as np

# 1. SETUP
theta = 1e-4                                            # convergence threshold

# 2. STATES AND ACTIONS (the MDP)
n = 10
n_states = n * n
goal = n_states - 1                                     # (9, 9)
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]              # Up, Down, Left, Right
arrows = ["^", "v", "<", ">"]

def move(state, action):                                # where does a move lead? (off-grid = stay)
    row, col = divmod(state, n)
    row = max(0, min(n - 1, row + moves[action][0]))
    col = max(0, min(n - 1, col + moves[action][1]))
    return row * n + col

# Transition model P[state][action] = list of (probability, next_state, reward)
P = {}
for s in range(n_states - 1):                           # goal is terminal: no actions
    P[s] = {}
    for a in range(4):
        outcomes = []
        for actual in range(4):
            prob = 0.8 if actual == a else 0.2 / 3      # 0.8 intended, 0.2 split over the other 3
            s2 = move(s, actual)
            reward = 10 if s2 == goal else -1
            outcomes.append((prob, s2, reward))
        P[s][a] = outcomes

# 3. TRAINING LOOP: Value Iteration
def value_iteration(gamma, max_iter=5000):
    V = np.zeros(n_states)                              # V[goal] stays 0
    for i in range(max_iter):
        delta = 0
        for s in P:
            # EQUATION: V(s) = max_a sum_s' P * (r + gamma * V(s'))
            q_values = [sum(p * (r + gamma * V[s2]) for p, s2, r in P[s][a]) for a in range(4)]
            new_v = max(q_values)
            delta = max(delta, abs(new_v - V[s]))
            V[s] = new_v
        if delta < theta:
            break
    policy = [int(np.argmax([sum(p * (r + gamma * V[s2]) for p, s2, r in P[s][a]) for a in range(4)])) if s in P else -1
              for s in range(n_states)]
    return V, policy, i + 1

# 4. OUTPUT
for gamma in [0.9, 0.0, 1.0]:
    V, policy, iters = value_iteration(gamma)
    print(f"\n=== gamma = {gamma} (converged in {iters} sweeps) ===")
    print("Value function:\n", np.round(V.reshape(n, n), 1))
    print("Policy:")
    for row in range(n):
        print(" ".join("G" if row * n + col == goal else arrows[policy[row * n + col]] for col in range(n)))

## Lab 3. Taxi Fleet Management with Policy Iteration (and Value Iteration comparison)

**Assignment**: 7 zones (A to G). Actions: move to a neighbouring zone or stay. Rewards: pick up a passenger `+20`, move without a passenger `-1`, fail to find a passenger `-5`. Moves succeed with probability 0.7 (0.3 traffic delay keeps the taxi in place). Passenger availability varies by zone (A high, D low). Implement the MDP, policy evaluation, policy improvement, iterate until convergence, print the optimal policy and values, and compare with Value Iteration.

**Theory**
- **Policy Iteration** alternates two steps until the policy stops changing:
  1. **Policy evaluation**: sweep `V(s) = sum P * [ r + gamma * V(s') ]` for the *current* policy until `delta < theta`.
  2. **Policy improvement**: set `policy(s) = argmax over a of sum P * [ r + gamma * V(s') ]`.
- **Value Iteration** (Lab 2) merges both into one update with a `max`.

**MDP design (state your design in the answer)**
- State = `(zone, passenger_available)`, so 14 states. Action = destination zone (own zone = stay).
- *Stay*: if a passenger is available, pick up `+20`; otherwise `-5`.
- *Move*: reward `-1`; with probability 0.7 arrive at the target, 0.3 stay.
- After any action, the passenger availability in the (new) zone is drawn with that zone's demand probability (A 0.9, D 0.1, ...).
- `gamma = 0.9`.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | a table `policy[(zone, passenger)]` that is improved |
| ENVIRONMENT | `P[state][action]` list of `(prob, next_state, reward)` |
| EQUATION | evaluation sweep, then greedy improvement with `argmax` |

**Observation to write (from my run)**
- Policy: stay in a zone when a passenger is there (+20). With no passenger, head toward the high-demand zone A (B and C go to A; D, E go to C; F, G go to E), and stay in A (demand 0.9) even when there is no passenger.
- Both algorithms gave the **same optimal policy and values**. Policy Iteration finished in about 4 improvement rounds (each with an inner evaluation loop); Value Iteration needed about 147 sweeps.
- Intuition: Policy Iteration usually needs far fewer outer iterations because each round evaluates a policy fully. Value Iteration does only one cheap backup per sweep, so it needs more sweeps, but each sweep is cheaper. Policy Iteration is better when evaluation is cheap or the state space is small; Value Iteration is simpler to code.

In [ ]:
import numpy as np

# 1. SETUP
gamma = 0.9
theta = 1e-6

# 2. STATES AND ACTIONS (the MDP)
zones = ["A", "B", "C", "D", "E", "F", "G"]
neighbors = {"A": ["B", "C"], "B": ["A", "D"], "C": ["A", "D", "E"], "D": ["B", "C", "F"],
             "E": ["C", "F", "G"], "F": ["D", "E", "G"], "G": ["E", "F"]}
demand = {"A": 0.9, "B": 0.5, "C": 0.6, "D": 0.1, "E": 0.4, "F": 0.3, "G": 0.5}   # P(passenger present)
actions = {z: [z] + neighbors[z] for z in zones}        # action = destination zone (own zone = stay)

# state = (zone, passenger_available) -> index
states = [(z, p) for z in zones for p in (0, 1)]
index = {s: i for i, s in enumerate(states)}
n_states = len(states)

# Transition model P[state][action] = list of (probability, next_state, reward)
def build_P():
    P = {}
    for (z, p) in states:
        P[(z, p)] = {}
        for a in actions[z]:
            if a == z:                                  # stay and look for a passenger
                reward = 20 if p == 1 else -5           # pick up (+20) or fail to find (-5)
                reached = [(1.0, z)]
            else:                                       # move without a passenger
                reward = -1
                reached = [(0.7, a), (0.3, z)]          # 0.7 arrive, 0.3 traffic keeps it here
            outcomes = []
            for prob_zone, z2 in reached:
                outcomes.append((prob_zone * demand[z2], (z2, 1), reward))
                outcomes.append((prob_zone * (1 - demand[z2]), (z2, 0), reward))
            P[(z, p)][a] = outcomes
    return P
P = build_P()

def q_value(V, s, a):
    return sum(prob * (r + gamma * V[index[s2]]) for prob, s2, r in P[s][a])

# 3. TRAINING LOOP: Policy Iteration
V = np.zeros(n_states)
policy = {s: s[0] for s in states}                      # start: always stay
pi_rounds = 0
while True:
    # EQUATION part 1: policy evaluation
    while True:
        delta = 0
        for s in states:
            new_v = q_value(V, s, policy[s])
            delta = max(delta, abs(new_v - V[index[s]]))
            V[index[s]] = new_v
        if delta < theta:
            break
    # EQUATION part 2: policy improvement
    stable = True
    for s in states:
        best = max(actions[s[0]], key=lambda a: q_value(V, s, a))
        if q_value(V, s, best) > q_value(V, s, policy[s]) + 1e-9:
            policy[s] = best
            stable = False
    pi_rounds += 1
    if stable:
        break

# Value Iteration (for comparison)
V_vi = np.zeros(n_states)
vi_sweeps = 0
while True:
    delta = 0
    for s in states:
        new_v = max(q_value(V_vi, s, a) for a in actions[s[0]])
        delta = max(delta, abs(new_v - V_vi[index[s]]))
        V_vi[index[s]] = new_v
    vi_sweeps += 1
    if delta < theta:
        break
vi_policy = {s: max(actions[s[0]], key=lambda a: q_value(V_vi, s, a)) for s in states}

# 4. OUTPUT
print("Policy Iteration took", pi_rounds, "improvement rounds | Value Iteration took", vi_sweeps, "sweeps")
print("\nZone | passenger? | best action | V (policy iter.) | V (value iter.)")
for s in states:
    act = "stay" if policy[s] == s[0] else "go " + policy[s]
    print(f"  {s[0]}  |     {s[1]}      | {act:6s}      | {V[index[s]]:8.2f}         | {V_vi[index[s]]:8.2f}")
print("\nSame policy from both methods:", all(policy[s] == vi_policy[s] for s in states))

## Lab 4. Monte Carlo Prediction (3-State MDP and 4x4 Grid World)

**Assignment, part 1**: a 3-state MDP (S1, S2, S3), actions Left/Right, fixed policy "always Right", `gamma = 0.9`. Use first-visit Monte Carlo for at least 1000 episodes and print `V`.
**Part 2**: 4x4 grid world, S0 to S15, start S0, terminal S15. Intended move with probability 0.8, otherwise a random other direction; off-grid keeps the state with reward `-1`; reaching S15 gives `+10`; all other moves give `0`. Uniformly random policy, `gamma = 0.95`, at least 10,000 episodes, print `V` as a 4x4 table.

**Theory**
- **First-visit Monte Carlo**: run a full episode, compute the return `G` backward (`G = r + gamma * G`), and for each state use the `G` from its **first** occurrence. Average these over episodes: `V[s] += (G - V[s]) / N[s]`.
- No bootstrapping, no model needed, only works for episodes that end (S3 and S15 are terminal).
- Part 1 exact answer (so you can verify): `V(S2) = 1`, `V(S1) = 0.9 * 1 = 0.9`, `V(S3) = 0`.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | part 1: always Right. Part 2: uniformly random |
| ENVIRONMENT | part 1: lookup table of `(next_state, reward)`. Part 2: 0.8 / 0.2 stochastic `step()` |
| EQUATION | after the episode: backward `G`, first-visit check, `V[s] += (G - V[s]) / N[s]` |

**Notes**
- Part 1 starts each episode at S1 or S2 so both states get visits (S3 is terminal). If the question fixes the start at S1, `V(S2)` would only be estimated when the episode passes through it, which it does anyway.
- Part 2 has a `max_steps` safety limit. A random walk can be long, and `gamma^1000` is practically zero, so cutting it off changes nothing.
- Part 2 results (my run): values grow from about `-2.6` at S0 to about `+3.6` next to the goal. S0 is negative because random walking into walls costs `-1` each time and the goal is far away.

In [ ]:
import numpy as np
import random

# ---------------- Part 1: the 3-state MDP ----------------
# 1. SETUP
gamma = 0.9
episodes = 2000

# 2. STATES AND ACTIONS
# transitions[state][action] = (next_state, reward)      S3 is terminal (it stays in S3 with reward 0)
transitions = {
    "S1": {"Right": ("S2", 0), "Left": ("S1", -1)},
    "S2": {"Right": ("S3", 1), "Left": ("S1", 0)},
}
V = {"S1": 0.0, "S2": 0.0, "S3": 0.0}
N = {"S1": 0, "S2": 0, "S3": 0}

# 3. TRAINING LOOP
for episode in range(episodes):
    state = random.choice(["S1", "S2"])                 # start anywhere so every state is visited
    history = []
    while state != "S3":
        # POLICY: always Right
        action = "Right"
        # ENVIRONMENT
        next_state, reward = transitions[state][action]
        history.append((state, reward))
        state = next_state

    # EQUATION: first-visit Monte Carlo (after the episode)
    first = {}
    for t, (s, r) in enumerate(history):
        first.setdefault(s, t)
    G = 0
    for t in reversed(range(len(history))):
        s, r = history[t]
        G = r + gamma * G
        if first[s] == t:                               # first visit of s
            N[s] += 1
            V[s] += (G - V[s]) / N[s]

# 4. OUTPUT
print("Part 1 estimated V:", {s: round(v, 3) for s, v in V.items()})
print("Exact V:            {'S1': 0.9, 'S2': 1.0, 'S3': 0.0}")

# ---------------- Part 2: 4x4 Gridworld, random policy ----------------
# 1. SETUP
gamma = 0.95
episodes = 10000
max_steps = 1000                                        # safety limit for very long random walks

# 2. STATES AND ACTIONS
n = 4
n_states = n * n
goal = 15
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]              # Up, Down, Left, Right
V = np.zeros(n_states)
N = np.zeros(n_states)

def step(state, action):                                # ENVIRONMENT
    if random.random() >= 0.8:                          # 0.2: a random OTHER direction
        action = random.choice([a for a in range(4) if a != action])
    row, col = divmod(state, n)
    new_row, new_col = row + moves[action][0], col + moves[action][1]
    if not (0 <= new_row < n and 0 <= new_col < n):     # off the grid: stay, reward -1
        return state, -1
    next_state = new_row * n + new_col
    return next_state, (10 if next_state == goal else 0)

# 3. TRAINING LOOP
for episode in range(episodes):
    state = 0
    history = []
    while state != goal and len(history) < max_steps:
        action = random.randint(0, 3)                   # POLICY: uniformly random
        next_state, reward = step(state, action)        # ENVIRONMENT
        history.append((state, reward))
        state = next_state

    # EQUATION: first-visit Monte Carlo
    first = {}
    for t, (s, r) in enumerate(history):
        first.setdefault(s, t)
    G = 0
    for t in reversed(range(len(history))):
        s, r = history[t]
        G = r + gamma * G
        if first[s] == t:
            N[s] += 1
            V[s] += (G - V[s]) / N[s]

# 4. OUTPUT
print("\nPart 2 estimated V (4x4):\n", np.round(V.reshape(n, n), 2))

## Lab 5. Delivery Drone with Tabular Q-Learning (16x16 grid)

**Assignment**: custom environment: 16x16 grid, start (0,0), battery 40, recharge stations (4,4) and (11,11), package 1 (pickup (0,15), drop-off (15,0)), package 2 (pickup (15,15), drop-off (7,12), only after package 1 is delivered), wind in rows/columns 6 to 9 (25% chance to drift down), obstacles at `r in {3,4,5}, c = 8` and `r = 10, c in {2..5}`. Rewards: step `-1` (and `-1` battery), wall/obstacle `-10`, recharge `+5`, package 1 delivered `+100`, package 2 delivered `+300` (terminal), battery 0 `-200` (terminal). Use a Q-dictionary, `eps_t = max(eps_min, eps_0 * exp(-lambda * t))` with `eps_0 = 1, eps_min = 0.01, lambda = 0.00008`, `gamma = 0.99`, `alpha = 0.1`, 10,000 episodes, and every 500 episodes run 100 greedy evaluation episodes and print mean reward and completion rate.

**Theory**
- **Q-Learning** (off-policy): `Q[s,a] += alpha * (r + gamma * max Q[s2] - Q[s,a])`.
- The state must include everything that changes the future: `(row, col, battery, stage)`, where `stage` says which package task is next (0 go to pickup 1, 1 carrying 1, 2 go to pickup 2, 3 carrying 2). That is how "package 2 only after package 1" is enforced.
- `t` in the epsilon formula is the **global step count** (it is `episode` in some write-ups; say which you used).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy with exponential decay of epsilon |
| ENVIRONMENT | `step(state, action)`: wind, walls, obstacles, battery, recharge, packages, terminals |
| EQUATION | Q-Learning with `best_next = 0` at terminal states |

**Two things to know (and mention in the viva)**
1. **Reward loophole**: the literal rule "recharge gives `+5` if `b < 40`" lets the drone step off a station and back on forever (net `+3` per two steps) instead of delivering. The code uses `recharge_bonus_below = 20` so the bonus is paid only for a genuinely low battery. Set it to `B_max` to see the literal behaviour (the agent farms the bonus).
2. **Honest result**: tabular Q-Learning with a battery dimension, sparse rewards and a 70-step task rarely completes both deliveries in 10,000 episodes. In my run the mean evaluation reward improved (about `-250` to about `-175`) but the completion rate stayed `0.00`. The code coarsens the battery (`battery // 5`) to shrink the table. Ways to improve (to write as observations): more episodes, reward shaping (small reward for picking up a package), a slower epsilon decay, or function approximation (DQN).

**Common mistakes**: forgetting that the drone stays in place on a collision (and still loses battery); not zeroing the future value at terminal states.

In [ ]:
import numpy as np
import random
from collections import defaultdict

# 1. SETUP
alpha = 0.1
gamma = 0.99
eps0, eps_min, lam = 1.0, 0.01, 0.00008
episodes = 10000
max_steps = 200                                         # safety limit per episode
recharge_bonus_below = 20                               # literal spec is B_max (40); see note in the markdown
eval_every = 500

# 2. STATES AND ACTIONS
n = 16
B_max = 40
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]              # 0 Up, 1 Down, 2 Left, 3 Right
stations = [(4, 4), (11, 11)]
pick1, drop1 = (0, 15), (15, 0)
pick2, drop2 = (15, 15), (7, 12)
obstacles = {(3, 8), (4, 8), (5, 8), (10, 2), (10, 3), (10, 4), (10, 5)}

# state = (row, col, battery, stage)
# stage 0: go to pickup 1 | 1: carrying package 1 | 2: package 1 delivered, go to pickup 2 | 3: carrying package 2
Q = defaultdict(float)                                  # Q[(state, action)] -> scalar

def step(state, action):                                # ENVIRONMENT
    r, c, b, stage = state
    dr, dc = moves[action]
    if 6 <= r <= 9 and 6 <= c <= 9 and random.random() < 0.25:
        dr, dc = 1, 0                                   # wind drifts the drone 1 cell down
    nr, nc = r + dr, c + dc
    b -= 1                                              # every move costs 1 battery
    reward = -1
    done = False

    if not (0 <= nr < n and 0 <= nc < n) or (nr, nc) in obstacles:
        nr, nc = r, c                                   # wall / obstacle: stay in place
        reward = -10
    else:
        if (nr, nc) in stations:
            if b < recharge_bonus_below:
                reward += 5                             # recharge bonus (only when battery is low)
            b = B_max
        if stage == 0 and (nr, nc) == pick1:
            stage = 1
        elif stage == 1 and (nr, nc) == drop1:
            stage = 2
            reward += 100
        elif stage == 2 and (nr, nc) == pick2:
            stage = 3
        elif stage == 3 and (nr, nc) == drop2:
            reward += 300
            done = True                                 # terminal goal

    if b <= 0 and not done:
        reward -= 200                                   # out of battery
        done = True
    return (nr, nc, b, stage), reward, done

def key(state):                                         # coarse battery level keeps the table small
    return (state[0], state[1], state[2] // 5, state[3])

def greedy(state):
    return int(np.argmax([Q[(key(state), a)] for a in range(4)]))

def evaluate(games=100):                                # greedy policy, epsilon = 0
    total, finished = 0, 0
    for _ in range(games):
        state, ep_reward = (0, 0, B_max, 0), 0
        for _ in range(max_steps):
            state, reward, done = step(state, greedy(state))
            ep_reward += reward
            if done:
                finished += reward > 250             # only the final delivery gives +300
                break
        total += ep_reward
    return total / games, finished / games

# 3. TRAINING LOOP
t = 0                                                   # global step counter for epsilon decay
for episode in range(1, episodes + 1):
    state = (0, 0, B_max, 0)
    for _ in range(max_steps):
        # POLICY: epsilon-greedy with exponential decay
        epsilon = max(eps_min, eps0 * np.exp(-lam * t))
        t += 1
        if random.random() < epsilon:
            action = random.randint(0, 3)
        else:
            action = greedy(state)

        # ENVIRONMENT
        next_state, reward, done = step(state, action)

        # EQUATION: Q-Learning (Bellman update)
        best_next = 0 if done else max(Q[(key(next_state), a)] for a in range(4))
        Q[(key(state), action)] += alpha * (reward + gamma * best_next - Q[(key(state), action)])

        state = next_state
        if done:
            break

    if episode % eval_every == 0:
        mean_reward, completion = evaluate()
        print(f"Episode {episode:5d} | epsilon {epsilon:.3f} | mean reward {mean_reward:8.1f} | completion {completion:.2f}")

# 4. OUTPUT (printed above every 500 episodes)
print("Q-table entries learned:", len(Q))

## Lab 6. SARSA Traffic-Light Controller

**Assignment**: a four-way intersection; vehicles arrive randomly; the agent decides to keep the current light green or switch. Minimize average waiting time and queue length. Implement SARSA with epsilon-greedy, a Q-table, Poisson arrivals, train over episodes (simulated days), compare with a fixed-cycle baseline, and compare training for different numbers of episodes (5000 and 10000).

**Theory**
- **SARSA** is on-policy: `Q[s,a] += alpha * (r + gamma * Q[s2,a2] - Q[s,a])`, where `a2` is the action actually chosen next by epsilon-greedy.
- **Poisson arrivals**: `np.random.poisson(rate)` returns how many cars arrive on each lane in one time step.
- **Average waiting time** is computed as total queue-steps divided by total cars (Little's law idea: each waiting car adds 1 to the queue for each step it waits).

**MDP design (state it in your answer)**
- State: `(level of the green lane's queue, level of the longest other queue)`, each discretized into 4 levels. That is only 16 states, so the Q-table learns fast.
- Actions: `0 = keep green`, `1 = switch green to the lane with the longest queue`.
- Reward: minus the total number of waiting cars. Switching wastes one step (no cars leave, lost time), which stops pointless flip-flopping.
- Baseline: fixed cycle N, E, S, W, 8 steps each.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy `choose(state)`, used twice (SARSA 1 and SARSA 2) |
| ENVIRONMENT | `step()`: serve cars, Poisson arrivals, reward = -total queue |
| EQUATION | SARSA with `Q[next_state][next_action]` |

**Observation to write (from my run)**: the fixed-cycle baseline had an average wait of about 12 steps per car (total queue about 31). SARSA reached about 6 steps per car (queue about 15), roughly half. SARSA after 5000 and after 10000 days gave nearly the same result (6.0 vs 6.2), which means it had already converged by 5000 episodes. More training only helps while Q is still changing. The small difference is just randomness.

In [ ]:
import numpy as np
import random
from collections import defaultdict

# 1. SETUP
alpha = 0.1
gamma = 0.95
epsilon = 0.1
day_length = 100                                        # time steps in one simulated day
arrival_rates = np.array([0.8, 0.5, 0.8, 0.5])          # Poisson mean arrivals per step: N, E, S, W
service = 4                                             # cars that can leave per step on a green lane

# 2. STATES AND ACTIONS
# action 0 = keep the current green, action 1 = switch green to the lane with the longest queue
n_actions = 2
Q = defaultdict(lambda: np.zeros(n_actions))            # Q-table

def level(q):                                           # discretize a queue length into 4 levels
    return 0 if q < 2 else 1 if q < 5 else 2 if q < 10 else 3

def longest_other(queues, green):                       # the lane (not green) with the longest queue
    others = [i for i in range(4) if i != green]
    return max(others, key=lambda i: queues[i])

def get_state(queues, green):                           # state = (green lane level, longest other lane level)
    return (level(queues[green]), level(queues[longest_other(queues, green)]))

def to_lane(queues, green, action):                     # action -> which lane is green next
    return green if action == 0 else longest_other(queues, green)

def step(queues, green, lane):                          # ENVIRONMENT
    queues = queues.copy()
    if lane == green:                                   # keep green: cars leave
        queues[green] -= min(queues[green], service)
    green = lane                                        # switching wastes this step (lost time, no cars leave)
    queues += np.random.poisson(arrival_rates)          # new cars arrive (Poisson)
    reward = -queues.sum()                              # every waiting car costs 1 per step
    return queues, green, reward

def choose(state, eps):                                 # POLICY: epsilon-greedy
    if random.random() < eps:
        return random.randint(0, n_actions - 1)
    return int(np.argmax(Q[state]))

# 3. TRAINING LOOP: SARSA
def train(episodes):
    Q.clear()
    for episode in range(episodes):
        queues, green = np.zeros(4, dtype=int), 0
        state = get_state(queues, green)
        action = choose(state, epsilon)                 # SARSA 1: first action before the loop
        for t in range(day_length):
            queues, green, reward = step(queues, green, to_lane(queues, green, action))   # ENVIRONMENT
            next_state = get_state(queues, green)
            next_action = choose(next_state, epsilon)                # SARSA 2: next action now
            # EQUATION: SARSA
            Q[state][action] += alpha * (
                reward + gamma * Q[next_state][next_action] - Q[state][action]
            )
            state, action = next_state, next_action

# Evaluation: average waiting time per car and average queue length over many days
def evaluate(policy, days=200):
    total_queue_steps, steps = 0, 0
    for _ in range(days):
        queues, green = np.zeros(4, dtype=int), 0
        for t in range(day_length):
            lane = policy(queues, green, t)
            queues, green, reward = step(queues, green, lane)
            total_queue_steps += -reward
            steps += 1
    expected_arrivals = days * day_length * arrival_rates.sum()
    return total_queue_steps / expected_arrivals, total_queue_steps / steps   # (wait per car, avg queue)

def fixed_cycle(queues, green, t):                      # baseline: rotate N->E->S->W, 8 steps each
    return (green + 1) % 4 if t % 8 == 7 else green

def sarsa_policy(queues, green, t):                     # greedy w.r.t. learned Q
    return to_lane(queues, green, int(np.argmax(Q[get_state(queues, green)])))

# 4. OUTPUT
wait, queue = evaluate(fixed_cycle)
print(f"Fixed-cycle baseline     | avg wait per car {wait:5.2f} steps | avg total queue {queue:5.2f}")
for episodes in [5000, 10000]:
    train(episodes)
    wait, queue = evaluate(sarsa_policy)
    print(f"SARSA after {episodes:5d} days | avg wait per car {wait:5.2f} steps | avg total queue {queue:5.2f}")

## Lab 7. Deep Q-Learning on CartPole

**Assignment**: implement a Python program using deep Q-learning to learn CartPole.

**Theory**
- CartPole state = 4 real numbers (cart position, cart velocity, pole angle, pole angular velocity). 2 actions (left, right). Reward `+1` per step the pole stays up (max 500).
- A table cannot hold continuous states, so a **neural network** maps `state -> [Q(left), Q(right)]`.
- DQN = Q-Learning + a network + two stabilizers: a **replay buffer** (train on random past transitions, breaks correlation) and a **target network** (a slowly refreshed copy used to compute the target, so the target does not move every step).
- Epsilon starts at 1.0 and decays to 0.05.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy on `model(x)` |
| ENVIRONMENT | `env.step(action)` |
| EQUATION | `loss = (Q(s,a) - (r + gamma * max Q_target(s2) * (1 - terminated)))^2` |

**Keep / change vs Lab 5 (Q-Learning)**: keep the target `r + gamma * max Q(next)`. Change the table to a network, the update to a loss with `zero_grad / backward / step`, and add the replay buffer and target network. `gather(1, actions)` picks the Q-value of the action actually taken.

The code is wrapped in a function `train_dqn(episodes)` so Lab 9 can reuse it as a baseline. **Expected output**: average reward rises from about 20 to 150+ (noisy).

In [ ]:
!pip install -q gymnasium

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import random
from collections import deque

def train_dqn(episodes=300):
    # 1. SETUP
    gamma = 0.99
    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.99
    batch_size = 64
    lr = 0.001

    # 2. STATES AND ACTIONS
    env = gym.make("CartPole-v1")                       # state: 4 numbers, 2 actions
    n_actions = 2
    model = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, n_actions))
    target_model = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, n_actions))
    target_model.load_state_dict(model.state_dict())    # start as an exact copy
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    buffer = deque(maxlen=10000)                        # replay buffer
    totals = []

    # 3. TRAINING LOOP
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        while not done:
            # POLICY: epsilon-greedy
            if random.random() < epsilon:
                action = random.randint(0, n_actions - 1)
            else:
                action = model(torch.tensor(state, dtype=torch.float32)).argmax().item()

            # ENVIRONMENT
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buffer.append((state, action, reward, next_state, terminated))
            state = next_state
            total_reward += reward

            # EQUATION: DQN (learn from a random mini-batch)
            if len(buffer) >= batch_size:
                batch = random.sample(buffer, batch_size)
                states, actions, rewards, next_states, dones = zip(*batch)
                states = torch.tensor(np.array(states), dtype=torch.float32)
                actions = torch.tensor(actions).unsqueeze(1)
                rewards = torch.tensor(rewards, dtype=torch.float32)
                next_states = torch.tensor(np.array(next_states), dtype=torch.float32)
                dones = torch.tensor(dones, dtype=torch.float32)

                q_value = model(states).gather(1, actions).squeeze(1)
                with torch.no_grad():
                    target = rewards + gamma * target_model(next_states).max(1)[0] * (1 - dones)
                loss = nn.functional.mse_loss(q_value, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        epsilon = max(epsilon_min, epsilon * epsilon_decay)   # explore less over time
        if (episode + 1) % 10 == 0:
            target_model.load_state_dict(model.state_dict())  # refresh the target network

        totals.append(total_reward)
        if (episode + 1) % 20 == 0:
            print("Episode", episode + 1, "Avg reward (last 20):", sum(totals[-20:]) / 20)

    # 4. OUTPUT
    print("Avg reward, first 20 episodes:", sum(totals[:20]) / 20)
    print("Avg reward, last 20 episodes: ", sum(totals[-20:]) / 20)
    return totals

dqn_totals = train_dqn(300)

## Lab 8. TD-Gammon: Temporal Difference Learning, Hyper-parameters, 1-ply vs 2-ply

**Assignment**: implement a program that learns the game with TD-Gammon style temporal difference learning. Try different hyper-parameters and observe the difference between **1-ply** (one move by the agent) and **2-ply** (one move by the agent plus one by the opponent) search in move selection. The reference repo has `net.py` (the neural network that evaluates positions) and `train.py` (the training loop that updates the weights).

**Theory**
- TD-Gammon learns a **value network** `V(board)` = probability that player X wins, using **self-play** and **TD(lambda)**. The only real reward is at the end (win 1, loss 0, draw 0.5); everything before is learned by bootstrapping from the next position's value.
- **TD error**: `delta = V(next position) - V(current position)`.
- **Eligibility trace**: `trace = gamma * lambda * trace + gradient of V(current)`. The weight update is `w += alpha * delta * trace`. `lambda = 0` is TD(0); `lambda = 1` is Monte Carlo.
- **1-ply**: for every legal move, evaluate the position right after my move with the network and pick the best (X maximizes, O minimizes).
- **2-ply**: for every legal move, also let the opponent reply with their best move, and judge the position after the reply. It looks further ahead, so it catches traps that 1-ply misses. It costs roughly (number of moves) times more evaluations.
- Backgammon is too large for an exam, so this uses **Tic-Tac-Toe with the same algorithm** (a 9-16-1 network written in NumPy, so no deep learning library is needed). Mention that the real TD-Gammon uses the same idea with a larger network and backgammon rules.

**Mapping to the reference files**
| Reference file | In this notebook |
|----------------|------------------|
| `net.py` (network + position evaluation) | `init_net`, `value_and_grads`, `board_value` |
| `train.py` (training loop) | the `train(alpha, lam)` function: self-play with TD(lambda) |

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | epsilon-greedy over 1-ply move values |
| ENVIRONMENT | place a mark, check win/draw, outcome only at the end |
| EQUATION | `delta = target - prev_value`; `w += alpha * delta * trace`; `trace = gamma*lambda*trace + grad` |

**Observation to write (from my run, win rate against a random opponent over 300 games)**
- `alpha = 0.1` with `lambda` 0, 0.7 or 0.9 all reached about 94 to 95% wins with 1-ply: in this small game lambda matters little because games are short.
- A small learning rate (`alpha = 0.01`) learned much more slowly (about 83%).
- 2-ply was equal or slightly better than 1-ply (about 96 to 97% vs 94 to 95%) with the same trained weights, because it sees the opponent's reply. The extra lookahead is worth most when the evaluation network is still imperfect.

In [ ]:
import numpy as np
import random

# 1. SETUP
gamma = 1.0
epsilon = 0.1
episodes = 3000
hidden = 16

# 2. STATES AND ACTIONS (Tic-Tac-Toe stands in for backgammon; the algorithm is identical)
lines = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]

def encode(board):                                      # board string -> 9 numbers (X=+1, O=-1, empty=0)
    return np.array([1.0 if c == "X" else -1.0 if c == "O" else 0.0 for c in board])

def play(board, square, player):
    return board[:square] + player + board[square + 1:]

def winner(board):
    for a, b, c in lines:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return None

def game_over(board):
    return winner(board) is not None or " " not in board

def legal(board):
    return [i for i in range(9) if board[i] == " "]

def other(player):
    return "O" if player == "X" else "X"

# --- net.py part: a tiny neural network V(board) = P(X wins), weights = [W1, b1, W2, b2]
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def init_net():
    return [np.random.randn(hidden, 9) * 0.1, np.zeros(hidden), np.random.randn(hidden) * 0.1, np.zeros(1)]

def value_and_grads(w, x):                              # forward pass + gradient of V w.r.t. every weight
    h = sigmoid(w[0] @ x + w[1])
    v = sigmoid(w[2] @ h + w[3][0])
    dv = v * (1 - v)
    dh = dv * w[2] * h * (1 - h)
    return v, [np.outer(dh, x), dh, dv * h, np.array([dv])]

def board_value(w, board):                              # true result if the game is over, else the network
    win = winner(board)
    if win == "X":
        return 1.0
    if win == "O":
        return 0.0
    if " " not in board:
        return 0.5
    return value_and_grads(w, encode(board))[0]

# --- move selection: 1-ply vs 2-ply search
def choose_move(w, board, player, ply):
    scores = []
    moves = legal(board)
    for m in moves:
        after = play(board, m, player)
        if ply == 2 and not game_over(after):           # 2-ply: also let the opponent reply
            opp = other(player)
            replies = [board_value(w, play(after, r, opp)) for r in legal(after)]
            scores.append(min(replies) if opp == "O" else max(replies))   # opponent picks what is best for it
        else:                                           # 1-ply: judge the position right after my move
            scores.append(board_value(w, after))
    best = max(scores) if player == "X" else min(scores)
    return random.choice([m for m, s in zip(moves, scores) if abs(s - best) < 1e-9])

# 3. TRAINING LOOP: self-play with TD(lambda)   (train.py part)
def train(alpha, lam):
    w = init_net()
    for episode in range(episodes):
        board, player = " " * 9, "X"
        traces = [np.zeros_like(p) for p in w]
        prev_value = None
        done = False
        while not done:
            # POLICY: epsilon-greedy, 1-ply
            if random.random() < epsilon:
                action = random.choice(legal(board))
            else:
                action = choose_move(w, board, player, ply=1)

            # ENVIRONMENT
            board = play(board, action, player)
            win = winner(board)
            done = game_over(board)

            # EQUATION: TD(lambda)
            v, grads = value_and_grads(w, encode(board))
            target = (1.0 if win == "X" else 0.0 if win == "O" else 0.5) if done else v
            if prev_value is not None:
                delta = gamma * target - prev_value     # TD error
                for i in range(4):
                    w[i] += alpha * delta * traces[i]   # w += alpha * delta * trace
            for i in range(4):
                traces[i] = gamma * lam * traces[i] + grads[i]
            prev_value = v
            player = other(player)
    return w

def win_rate_vs_random(w, ply, games=300):              # X = trained agent, O = random
    wins = draws = 0
    for _ in range(games):
        board, player = " " * 9, "X"
        while not game_over(board):
            m = choose_move(w, board, "X", ply) if player == "X" else random.choice(legal(board))
            board = play(board, m, player)
            player = other(player)
        wins += winner(board) == "X"
        draws += winner(board) is None
    return wins / games, draws / games

# 4. OUTPUT: different hyper-parameters, 1-ply vs 2-ply
configs = [(0.1, 0.7), (0.1, 0.0), (0.1, 0.9), (0.01, 0.7)]    # (alpha, lambda)
print("alpha  lambda | 1-ply: win / draw | 2-ply: win / draw")
for alpha, lam in configs:
    w = train(alpha, lam)
    w1, d1 = win_rate_vs_random(w, ply=1)
    w2, d2 = win_rate_vs_random(w, ply=2)
    print(f"{alpha:5}  {lam:5} |   {w1:.2f} / {d1:.2f}     |   {w2:.2f} / {d2:.2f}")

## Lab 9. REINFORCE on CartPole, Compared with a Deep Q-Learning Baseline

**Assignment**: implement REINFORCE to train CartPole and evaluate its performance against a Deep Q-Learning baseline.

**Theory**
- Value-based methods (DQN) learn `Q` and act greedily. **Policy-gradient** methods learn the policy itself: a network that outputs action probabilities.
- **REINFORCE**: play a whole episode, then make actions that led to a high return more likely and low-return actions less likely: `loss = -log_prob(action) * G`.
- It is Monte Carlo applied to a policy: it uses the full return `G` and updates after the episode. Normalizing the returns (subtract mean, divide by std) acts as a baseline and reduces variance.

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | `Categorical(actor(x)).sample()`, save `log_prob` |
| ENVIRONMENT | `env.step(action.item())`, save `reward` |
| EQUATION | after the episode: backward `G`, normalize, `loss = -(log_probs * returns).mean()` |

**REINFORCE vs DQN (what to write)**
| | REINFORCE | DQN |
|---|-----------|-----|
| Learns | the policy directly | action values `Q`, policy is `argmax` |
| Updates | after each episode (Monte Carlo) | every step from a replay buffer |
| Exploration | sampling from the policy (stochastic) | epsilon-greedy |
| Sample efficiency | lower (data used once) | higher (replay reuses data) |
| Variance | high (hence normalization) | lower, but can be unstable |
| Works with continuous actions | yes (Gaussian policy) | not directly |

The second cell runs the DQN baseline (the `train_dqn` function from Lab 7) and plots both. **Run the Lab 7 cell first.** Expected: both rise, REINFORCE is noisier from episode to episode, and DQN often learns faster early on.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def train_reinforce(episodes=500):
    # 1. SETUP
    gamma = 0.99
    lr = 0.01

    # 2. STATES AND ACTIONS
    env = gym.make("CartPole-v1")                       # state: 4 numbers, 2 actions
    n_actions = 2
    actor = nn.Sequential(
        nn.Linear(4, 128), nn.ReLU(),
        nn.Linear(128, n_actions), nn.Softmax(dim=-1)
    )
    optimizer = torch.optim.Adam(actor.parameters(), lr=lr)
    totals = []

    # 3. TRAINING LOOP
    for episode in range(episodes):
        state, _ = env.reset()
        log_probs = []
        rewards = []
        done = False

        while not done:
            x = torch.tensor(state, dtype=torch.float32)

            # POLICY: actor
            dist = torch.distributions.Categorical(actor(x))
            action = dist.sample()
            log_probs.append(dist.log_prob(action))

            # ENVIRONMENT
            state, reward, terminated, truncated, _ = env.step(action.item())
            rewards.append(reward)
            done = terminated or truncated

        # EQUATION: REINFORCE (after the episode)
        G = 0
        returns = []
        for reward in reversed(rewards):
            G = reward + gamma * G
            returns.insert(0, G)

        returns = torch.tensor(returns, dtype=torch.float32)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = -(torch.stack(log_probs) * returns).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        totals.append(sum(rewards))
        if (episode + 1) % 50 == 0:
            print("Episode", episode + 1, "Avg reward (last 50):", sum(totals[-50:]) / 50)

    # 4. OUTPUT
    print("Avg reward, last 50 episodes:", sum(totals[-50:]) / 50)
    return totals

reinforce_totals = train_reinforce(500)

# ---- Compare with the Deep Q-Learning baseline (train_dqn comes from the Lab 7 cell) ----
dqn_totals = train_dqn(500)

def moving_average(x, k=20):
    return [sum(x[max(0, i - k):i + 1]) / len(x[max(0, i - k):i + 1]) for i in range(len(x))]

print("Last-50 average -> REINFORCE:", sum(reinforce_totals[-50:]) / 50, "| DQN:", sum(dqn_totals[-50:]) / 50)
print("Best episode    -> REINFORCE:", max(reinforce_totals), "| DQN:", max(dqn_totals))
plt.plot(moving_average(reinforce_totals), label="REINFORCE")
plt.plot(moving_average(dqn_totals), label="DQN")
plt.xlabel("Episode"); plt.ylabel("Reward (moving average)"); plt.legend(); plt.show()

## Lab 10. Elevator Control with Actor-Critic (Discrete and Continuous Actions)

**Assignment**: train an elevator control agent with Actor-Critic. The agent must learn discrete actions (up or down) and continuous decisions (elevator speed and timing), minimizing passenger waiting time.

**Theory**
- **Actor** = the policy, **Critic** = the value function. The critic's TD error is the **advantage**: `advantage = r + gamma * V(s') - V(s)`. The actor is pushed toward actions with a positive advantage.
- **Hybrid action space**: the actor has a shared body and two heads.
  - **Discrete head**: logits over {down, up} gives a `Categorical` distribution.
  - **Continuous head**: a mean for (speed, dwell time) squeezed to `[0, 1]` with `sigmoid`, plus a learnable `log_std`, gives a `Normal` distribution.
  - The joint log-probability is `log_prob(direction) + log_prob(speed, dwell)`, because the two choices are independent.
- It updates **every step** (one-step TD), so it learns online.

**Environment (the question leaves it open, so a simple simulation is the right answer)**
- State: elevator position plus the number of passengers waiting on each of 5 floors (6 numbers).
- Action: direction (discrete) + speed and door-dwell time (continuous, both in `[0, 1]`). Speed sets how far the elevator moves; dwell time sets how many people board when it stops at a floor but also costs time.
- Reward: `-(waiting passengers x time used) / 100 - 0.2 * speed^2` (waiting cost plus a small energy cost).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | sample `direction` and `(speed, dwell)` from the two heads |
| ENVIRONMENT | `env.step(direction, speed, dwell)` |
| EQUATION | `loss = -log_prob * advantage.detach() + advantage ** 2` |

**Keep / change vs Lab 9**: keep `Categorical`/`log_prob`. Add the critic, add the continuous `Normal` head, and update every step. **Expected output**: the average waiting cost falls compared with the first 30 episodes and is below the random controller's cost.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.001
episodes = 300

# 2. STATES AND ACTIONS: a simple elevator simulation
class ElevatorEnv:
    n_floors = 5
    arrival_rates = np.array([0.5, 0.2, 0.2, 0.2, 0.5])       # passengers arriving per time unit on each floor

    def reset(self):
        self.pos = 0.0                                         # elevator position (0 to 4, continuous)
        self.waiting = np.zeros(self.n_floors)                 # passengers waiting on each floor
        self.t = 0
        self.wait_cost = 0.0
        return self.obs()

    def obs(self):                                             # 6 numbers: position + 5 queue sizes
        return np.concatenate(([self.pos / 4], self.waiting / 10)).astype(np.float32)

    def step(self, direction, speed, dwell):
        # direction: discrete (0 = down, 1 = up) | speed, dwell: continuous in [0, 1]
        self.pos = float(np.clip(self.pos + (1 if direction == 1 else -1) * speed * 2, 0, 4))
        floor = int(round(self.pos))
        if abs(self.pos - floor) < 0.3:                        # close enough to a floor: open the doors
            boarded = min(self.waiting[floor], int(dwell * 6)) # longer dwell time = more people board
            self.waiting[floor] -= boarded
        time_used = 1 + dwell                                  # dwelling costs time
        self.waiting += np.random.poisson(self.arrival_rates * time_used)
        cost = self.waiting.sum() * time_used                  # waiting passengers x time
        self.wait_cost += cost
        self.t += 1
        reward = -cost / 100 - 0.2 * speed ** 2                # minimize waiting (+ small energy cost for speed)
        return self.obs(), reward, self.t >= 100

env = ElevatorEnv()

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(6, 64), nn.ReLU())
        self.direction = nn.Linear(64, 2)                      # discrete head: up / down
        self.mean = nn.Linear(64, 2)                           # continuous head: mean of (speed, dwell)
        self.log_std = nn.Parameter(torch.full((2,), -1.0))    # learnable exploration noise
    def forward(self, x):
        h = self.body(x)
        dir_dist = torch.distributions.Categorical(logits=self.direction(h))
        cont_dist = torch.distributions.Normal(torch.sigmoid(self.mean(h)), self.log_std.exp())
        return dir_dist, cont_dist

actor = Actor()
critic = nn.Sequential(nn.Linear(6, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=lr)
costs = []

# 3. TRAINING LOOP
for episode in range(episodes):
    state = env.reset()
    done = False
    while not done:
        x = torch.tensor(state)

        # POLICY: discrete direction + continuous speed/dwell
        dir_dist, cont_dist = actor(x)
        direction = dir_dist.sample()
        cont = cont_dist.sample()
        log_prob = dir_dist.log_prob(direction) + cont_dist.log_prob(cont).sum()

        # ENVIRONMENT
        speed, dwell = cont.clamp(0, 1).tolist()
        next_state, reward, done = env.step(direction.item(), speed, dwell)

        # EQUATION: Actor-Critic (one-step TD error as the advantage)
        value = critic(x).squeeze()
        next_value = critic(torch.tensor(next_state)).squeeze().detach() * (0 if done else 1)
        advantage = reward + gamma * next_value - value
        loss = -log_prob * advantage.detach() + advantage ** 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        state = next_state

    costs.append(env.wait_cost)
    if (episode + 1) % 30 == 0:
        print("Episode", episode + 1, "Avg waiting cost (last 30):", round(sum(costs[-30:]) / 30, 1))

# 4. OUTPUT: compare with a random controller
random_costs = []
for _ in range(30):
    env.reset()
    done = False
    while not done:
        _, _, done = env.step(np.random.randint(2), np.random.rand(), np.random.rand())
    random_costs.append(env.wait_cost)
print("Random controller waiting cost:", round(sum(random_costs) / 30, 1))
print("Trained actor-critic (first 30 episodes):", round(sum(costs[:30]) / 30, 1), "| (last 30):", round(sum(costs[-30:]) / 30, 1))

## Lab 11. Autonomous Delivery Vehicle Fuel Efficiency: Simple vs Advanced Actor-Critic

**Assignment**: design an actor-critic agent that optimizes the fuel efficiency of an autonomous delivery vehicle by deciding when to accelerate, brake or maintain speed, given varying traffic conditions and road layouts. **Compare Simple Actor-Critic with Advanced Actor-Critic.** (Reference: tsmatz's actor-critic notebook.)

**Environment**
- A 50-unit road split into 10 segments. Each new episode draws a random **road layout** (slope per segment) and **traffic** (speed limit of 3, 6 or 9 per segment).
- State: `[speed, progress, current speed limit, current slope]`. Actions: 0 accelerate, 1 maintain, 2 brake.
- Fuel: grows with speed squared, with acceleration, and with uphill slope times speed. Reward: `-fuel - 0.1 (time) - speeding penalty`. So the agent must trade off fuel against time and traffic limits.

**Simple vs Advanced (what to write in your comparison)**
| | Simple Actor-Critic | Advanced Actor-Critic (A2C) |
|---|---------------------|-----------------------------|
| Networks | separate actor and critic | shared body, two heads |
| Update | every step, one-step TD target | once per episode, using returns |
| Advantage | `r + gamma*V(s') - V(s)` | `G - V(s)` |
| Extras | none | entropy bonus (explores), return normalization, gradient clipping |
| Variance / bias | low variance, biased by the critic | lower bias, steadier from run to run |

**Slots**
| Slot | Simple | Advanced |
|------|--------|----------|
| POLICY | `Categorical(actor(x))` | `Categorical(softmax(net(x)))` |
| ENVIRONMENT | `env.step(action)` | same |
| EQUATION | TD advantage, update each step | returns, advantage, actor + 0.5 critic - 0.01 entropy |

**Expected output**: a two-row table (average fuel, average trip time, completion rate over the last 50 episodes). The advanced version is usually steadier. Because RL is noisy, one run can reverse the ranking; say so in your observations.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.003
episodes = 400

# 2. STATES AND ACTIONS: an autonomous delivery vehicle on a 50-unit road
class VehicleEnv:
    def reset(self):
        self.pos, self.speed, self.t, self.fuel = 0.0, 0.0, 0, 0.0
        self.slope = np.random.uniform(-0.1, 0.1, 10)          # road layout: slope of 10 segments
        self.limit = np.random.choice([3, 6, 9], 10)           # traffic: speed limit of each segment
        return self.obs()

    def segment(self):
        return min(int(self.pos // 5), 9)

    def obs(self):                                             # speed, progress, traffic limit, slope
        s = self.segment()
        return np.array([self.speed / 10, self.pos / 50, self.limit[s] / 10, self.slope[s] * 10], dtype=np.float32)

    def step(self, action):                                    # 0 accelerate, 1 maintain, 2 brake
        accel = [1.0, 0.0, -1.0][action]
        self.speed = float(np.clip(self.speed + accel, 0, 10))
        s = self.segment()
        self.pos += self.speed
        fuel = 0.05 + 0.01 * self.speed ** 2 + 0.3 * max(accel, 0) + 0.5 * max(self.slope[s], 0) * self.speed
        self.fuel += fuel
        self.t += 1
        reward = -fuel - 0.1 - 1.0 * max(0, self.speed - self.limit[s])   # fuel + time + speeding penalty
        reached = self.pos >= 50
        return self.obs(), reward, reached or self.t >= 100, reached

env = VehicleEnv()
n_actions = 3

# 3. TRAINING LOOP
# ---------- SIMPLE actor-critic: separate networks, one-step TD update at every step ----------
def train_simple():
    actor = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, n_actions), nn.Softmax(dim=-1))
    critic = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 1))
    optimizer = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=lr)
    fuels = []
    for episode in range(episodes):
        state, done = env.reset(), False
        while not done:
            x = torch.tensor(state)
            dist = torch.distributions.Categorical(actor(x))                   # POLICY
            action = dist.sample()
            next_state, reward, done, reached = env.step(action.item())        # ENVIRONMENT
            value = critic(x).squeeze()                                        # EQUATION
            next_value = critic(torch.tensor(next_state)).squeeze().detach() * (0 if reached else 1)
            advantage = reward + gamma * next_value - value
            loss = -dist.log_prob(action) * advantage.detach() + advantage ** 2
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            state = next_state
        fuels.append((env.fuel, env.t, reached))
    return fuels

# ---------- ADVANCED actor-critic (A2C): shared body, episode returns, entropy bonus, grad clipping ----------
class SharedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(4, 64), nn.ReLU())
        self.actor = nn.Linear(64, n_actions)
        self.critic = nn.Linear(64, 1)
    def forward(self, x):
        h = self.body(x)
        return torch.softmax(self.actor(h), dim=-1), self.critic(h).squeeze(-1)

def train_advanced():
    net = SharedNet()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    fuels = []
    for episode in range(episodes):
        state, done = env.reset(), False
        log_probs, values, entropies, rewards = [], [], [], []
        while not done:
            probs, value = net(torch.tensor(state))
            dist = torch.distributions.Categorical(probs)                      # POLICY
            action = dist.sample()
            log_probs.append(dist.log_prob(action)); values.append(value); entropies.append(dist.entropy())
            state, reward, done, reached = env.step(action.item())             # ENVIRONMENT
            rewards.append(reward)
        G, returns = 0, []                                                     # EQUATION (after the episode)
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        values = torch.stack(values)
        advantage = returns - values
        loss = (-(torch.stack(log_probs) * advantage.detach()).mean()
                + 0.5 * (advantage ** 2).mean()
                - 0.01 * torch.stack(entropies).mean())
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        optimizer.step()
        fuels.append((env.fuel, env.t, reached))
    return fuels

simple = train_simple()
advanced = train_advanced()

# 4. OUTPUT: compare (last 50 episodes)
def summary(name, results):
    last = results[-50:]
    print(f"{name:16s} | avg fuel {np.mean([f for f, t, r in last]):5.2f} | avg time {np.mean([t for f, t, r in last]):5.1f} steps | trips completed {np.mean([r for f, t, r in last]):.2f}")
summary("Simple AC", simple)
summary("Advanced AC (A2C)", advanced)

## Lab 12. PPO for a Real-World Task (Simulated Inventory Management)

**Assignment**: apply PPO to any real-world task; a simulated environment is allowed. (Reference: "PPO from scratch" Kaggle notebook.)

**Task chosen: warehouse inventory management**
- Each day the agent orders 0, 5 or 10 units, then random demand (Poisson, about 4 a day) arrives. Reward = revenue `5 x sold` minus purchase cost `2 x ordered` minus a fixed order fee `3` minus a holding cost `0.2 x stock` minus a penalty `1 x lost sales`. An episode is one 30-day month.
- It is a good PPO task: stochastic, sequential, and there is a clear trade-off between over-ordering (holding cost) and under-ordering (lost sales).

**Theory**
- Plain policy gradient can take one step that wrecks the policy. **PPO** keeps updates small by clipping the probability ratio `ratio = pi_new(a|s) / pi_old(a|s) = exp(new_log_prob - old_log_prob)`:
  `loss = -min(ratio * A, clip(ratio, 0.8, 1.2) * A)`.
- It also lets the same batch of data be reused for several update epochs.
- Everything else is the actor-critic of Lab 11 (advantage `A = G - V`, critic loss, entropy bonus).

**Slots**
| Slot | This lab |
|------|----------|
| POLICY | actor samples an order size, **save the old log prob** (`.detach()`) |
| ENVIRONMENT | `InventoryEnv.step(action)` |
| EQUATION (after each batch, 4 epochs) | ratio, clipped surrogate, critic loss, entropy bonus |

**Three changes from A2C (marked `PPO 1/2/3` in the code)**: save old log probs; repeat the update several times on the same batch; use the clipped `min(...)` objective.

**Expected output**: average monthly profit for a random policy (about 150), a simple "order 10 when stock is low" rule (about 290), and the trained PPO. PPO should clearly beat random and approach the rule. Different numbers on different runs are normal.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# 1. SETUP
gamma = 0.99
lr = 0.003
clip_eps = 0.2
epochs = 4                                              # PPO updates per batch of data
updates = 60
episodes_per_update = 5

# 2. STATES AND ACTIONS: real-world task = warehouse inventory management (simulated)
class InventoryEnv:
    max_stock, horizon = 20, 30                         # 30-day month
    orders = [0, 5, 10]                                 # action 0/1/2 = order 0, 5 or 10 units

    def reset(self):
        self.stock, self.day = 10, 0
        return self.obs()

    def obs(self):
        return np.array([self.stock / self.max_stock, self.day / self.horizon], dtype=np.float32)

    def step(self, action):
        order = self.orders[action]
        self.stock = min(self.max_stock, self.stock + order)
        demand = np.random.poisson(4)                   # customers want ~4 units a day
        sold = min(self.stock, demand)
        lost = demand - sold                            # unmet demand
        self.stock -= sold
        self.day += 1
        reward = 5 * sold - 2 * order - 3 * (order > 0) - 0.2 * self.stock - 1 * lost
        return self.obs(), reward, self.day >= self.horizon   # revenue - purchase - fixed fee - holding - stock-out

env = InventoryEnv()
actor = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 3), nn.Softmax(dim=-1))
critic = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=lr)

# 3. TRAINING LOOP
for update in range(updates):
    xs, actions, old_log_probs, returns, ep_rewards = [], [], [], [], []

    for _ in range(episodes_per_update):                # collect a small batch of episodes
        state, done, rewards = env.reset(), False, []
        while not done:
            x = torch.tensor(state)

            # POLICY: actor
            dist = torch.distributions.Categorical(actor(x))
            action = dist.sample()
            xs.append(x); actions.append(action)
            old_log_probs.append(dist.log_prob(action).detach())            # PPO 1: save old log prob

            # ENVIRONMENT
            state, reward, done = env.step(action.item())
            rewards.append(reward)

        G, ep_returns = 0, []                           # discounted returns of this episode
        for r in reversed(rewards):
            G = r + gamma * G
            ep_returns.insert(0, G)
        returns += ep_returns
        ep_rewards.append(sum(rewards))

    # EQUATION: PPO (several updates on the same batch)
    xs, actions, old_log_probs = torch.stack(xs), torch.stack(actions), torch.stack(old_log_probs)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    for _ in range(epochs):                                                 # PPO 2: reuse the data
        dist = torch.distributions.Categorical(actor(xs))
        new_log_probs = dist.log_prob(actions)
        value = critic(xs).squeeze(-1)
        advantage = returns - value.detach()

        ratio = torch.exp(new_log_probs - old_log_probs)                    # PPO 2: probability ratio
        surr1 = ratio * advantage
        surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantage  # PPO 3: clip
        actor_loss = -torch.min(surr1, surr2).mean()                        # PPO 3: pessimistic min
        critic_loss = ((returns - value) ** 2).mean()
        loss = actor_loss + 0.5 * critic_loss - 0.01 * dist.entropy().mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (update + 1) % 10 == 0:
        print("Update", update + 1, "Avg monthly profit:", round(float(np.mean(ep_rewards)), 1))

# 4. OUTPUT: compare PPO with simple baselines
def evaluate(policy, n=100):
    totals = []
    for _ in range(n):
        state, done, total = env.reset(), False, 0
        while not done:
            state, reward, done = env.step(policy(state))
            total += reward
        totals.append(total)
    return np.mean(totals)

random_policy = lambda s: np.random.randint(3)
rule_policy = lambda s: 2 if s[0] * 20 < 6 else 0                           # simple rule: order 10 when stock < 6
ppo_policy = lambda s: int(torch.argmax(actor(torch.tensor(s))))
print("Avg monthly profit | random:", round(evaluate(random_policy), 1),
      "| order-when-low rule:", round(evaluate(rule_policy), 1),
      "| PPO:", round(evaluate(ppo_policy), 1))

## Quick Revision: Same Template, Different Slots

| # | Algorithm | POLICY | EQUATION slot | Extra |
|---|-----------|--------|---------------|-------|
| 1 | Epsilon-greedy variants + regret | epsilon from a decay function | `Q[a] += (r - Q[a]) / N[a]`; `regret += mu* - mu[a]` | no state, no gamma |
| 2 | Value Iteration | greedy on `V` (at the end) | `V[s] = max_a sum p*(r + g*V[s2])` | needs the model, `theta = 1e-4` |
| 3 | Policy Iteration | table `policy[s]` | evaluate sweeps, then `argmax` improvement | compare with Value Iteration |
| 4 | Monte Carlo prediction | fixed / random | `V[s] += (G - V[s]) / N[s]` after the episode | first-visit, no bootstrapping |
| 5 | Q-Learning (drone) | epsilon-greedy, exponential decay | `Q += a*(r + g*max Q[s2] - Q)` | dict Q, terminal handling |
| 6 | SARSA (traffic) | epsilon-greedy, twice | `Q += a*(r + g*Q[s2,a2] - Q)` | Poisson arrivals, baseline |
| 7 | DQN | epsilon-greedy on `model` | `(Q(s,a) - (r + g*max Q_target(s2)))^2` | replay buffer + target net |
| 8 | TD-Gammon | 1-ply / 2-ply on the network | `w += a*delta*trace` | TD(lambda), self-play |
| 9 | REINFORCE | sample from actor | `loss = -(log_prob * G)` after the episode | compare with DQN |
| 10 | Actor-Critic | discrete + continuous heads | `adv = r + g*V(s2) - V(s)` every step | hybrid action space |
| 11 | Simple vs Advanced AC | sample from actor | TD per step vs `adv = G - V` + entropy | comparison table |
| 12 | PPO | sample, save old log prob | `-min(ratio*adv, clip(ratio)*adv)` | 4 epochs on each batch |

**Family tree**
- `Lab 1` + state and `gamma*max Q[next]` -> `Lab 5` (Q-Learning) -> use `Q[next, next_action]` -> `Lab 6` (SARSA). `Lab 5` + network, replay, target network -> `Lab 7` (DQN).
- `Lab 3` and `Lab 2` are the planning pair (both need the model). `Lab 4` (MC) and `Lab 2` of the earlier template (TD(0)) are prediction. `Lab 8` is TD(0) with a network and traces.
- `Lab 4` (returns) applied to a policy -> `Lab 9` (REINFORCE) -> add critic -> `Lab 10` -> returns and entropy -> `Lab 11` (A2C) -> clipping and reuse -> `Lab 12` (PPO).

**Categories**
- Needs a model: Labs 2, 3. Model-free: everything else.
- On-policy: 6, 9, 10, 11, 12. Off-policy: 5, 7.
- Updates every step: 5, 6, 7, 8, 10, simple AC in 11. After the episode: 4, 9, advanced AC in 11, 12.

## Viva Questions and One-Line Answers

- **What is regret?** The total reward lost compared with always choosing the best arm: `sum(mu* - mu_chosen)`.
- **Why decay epsilon?** Explore a lot while you know little, then exploit once the estimates are good. It lowers regret.
- **Value Iteration vs Policy Iteration?** Value Iteration does one `max` backup per sweep and extracts the policy at the end. Policy Iteration fully evaluates a policy, then improves it greedily, repeating until stable. Both need a model and give the same optimum.
- **What does `gamma = 0` do? `gamma = 1`?** `gamma = 0` is fully myopic (only the immediate reward counts). `gamma = 1` counts all future rewards equally (only safe for episodic tasks that end).
- **TD vs Monte Carlo?** TD bootstraps each step (low variance, some bias). MC waits for the full return (unbiased, high variance, needs episodes that end).
- **Q-Learning vs SARSA?** Q-Learning uses `max` of the next state (off-policy). SARSA uses the actual next action (on-policy, accounts for exploration).
- **Why replay buffer and target network in DQN?** Replay breaks the correlation between samples; the target network keeps the learning target stable.
- **What is the eligibility trace in TD-Gammon?** A decaying record of which weights recently influenced the value, so a later TD error also credits earlier positions.
- **1-ply vs 2-ply?** 1-ply evaluates the position after my move. 2-ply also considers the opponent's best reply, so it looks further ahead at higher cost.
- **REINFORCE vs DQN?** REINFORCE learns the policy directly from full-episode returns; DQN learns Q-values and acts greedily. REINFORCE has higher variance and lower sample efficiency but handles stochastic and continuous policies.
- **How does an actor handle continuous actions?** It outputs the mean (and std) of a Normal distribution, samples from it, and uses its log-probability in the loss.
- **Simple vs advanced actor-critic?** Simple = one-step TD, updated every step. Advanced (A2C) adds returns-based advantage, a shared network, an entropy bonus and gradient clipping for stability.
- **What does PPO clipping do?** It limits how far the new policy can move from the old one in a single update.

## Last-Minute Checklist and Common Bugs

- [ ] Copy every number from the question into SETUP (alpha, gamma, thresholds, episodes, trials).
- [ ] State your assumptions in a comment or markdown (how you modelled "random other direction", what the MDP states are).
- [ ] Terminal states: loop stops, `V[terminal] = 0`, and the Q-target has no future term.
- [ ] SARSA: `next_action` chosen before the update; `action = next_action` at the end.
- [ ] Returns are computed **backward**: `for r in reversed(rewards): G = r + gamma * G`.
- [ ] Use `torch.stack(...)` for lists of tensors that need gradients; `.detach()` for targets, advantages in the actor loss, and old log probs.
- [ ] Gymnasium: `reset()` returns `(state, info)`; `step()` returns 5 values; use `action.item()`.
- [ ] Print the output the question asks for (value grid, policy arrows, mean reward, completion rate, comparison table) and write the observations in 2 to 3 sentences.
- [ ] RL is noisy. If a result looks off, say so, and fix seeds (`random.seed(0)`, `np.random.seed(0)`, `torch.manual_seed(0)`) if reproducibility is requested.